# Insider Disclosures and Stock Returns: Phase 3

**INFO 2950 | Vansh Kakra (vk365), Dev Shukla (ds2395), and Wilson Wang (ww576)**  
**Mentor:** Saif Mahmud (sm2446)

**Working draft:** The data-loading checks, data description, and original analysis plans are included below. The research models and significance tests have not been run. Findings, conclusions, and the final bibliography will be completed after those analyses.

## 1. Introduction

Company insiders disclose purchases and sales of their company's shares through SEC filings. These disclosures raise a question about market efficiency: once a trade becomes public, does subsequent stock performance contain information beyond broad market movements? Purchases and sales may have different motivations, so we examine them separately rather than treating every insider transaction as the same signal.

In Phase 2, we combined SEC transaction records, historical S&P 500 membership, and adjusted stock prices. We now use the cleaned event dataset to examine two questions:

1. **Are purchase events followed by positive 13-week abnormal returns, and are those returns higher than returns following sale events?**
2. **Among purchase events, are C-suite participation and larger dollar purchases associated with higher 13-week abnormal returns, controlling for the number of recorded insiders?**

An **event** is one company's purchases or sales grouped within a week. An **abnormal return** is the difference between a stock's actual return and its expected return based on its earlier relationship with SPY, our market benchmark. We sum these differences to obtain a **cumulative abnormal return (CAR)**. The primary outcome is the 13-week CAR; the 1-, 4-, and 26-week horizons provide secondary descriptive context.

The literature cited in the preregistrations motivates these questions but does not guarantee the same patterns in our sample. Our analysis concerns associations following public disclosures. It cannot establish insiders' motives or show that the trades caused later returns.

The introduction's summary of findings will be added after the planned analyses are complete.

## 2. Loading the Cleaned Data

The cleaning process is complete in [data_cleaning.ipynb](https://github.com/Vansh-Kakra/DSproject/blob/phase-3-analysis/insider-trading-phase3-4/data_cleaning.ipynb). This notebook starts from its exported CSV files. The code below only loads and checks those exports; it does not repeat the raw-data cleaning.

### 2.1 Imports and File Locations

We use the Phase 2 Python environment. Open this notebook from the repository root or the `insider-trading-phase3-4` folder, then restart the kernel and run the cells in order. The saved outputs below show the checks on the current data snapshot.

In [1]:
# Import libraries and locate the cleaned inputs from either supported working folder.
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "insider-trading-phase3-4" / "data" / "processed" / "insider_events.csv").is_file()), None)
if repo_root is None:
    raise FileNotFoundError("Open the notebook within the repository and restore insider-trading-phase3-4/data/processed.")
processed_dir = repo_root / "insider-trading-phase3-4" / "data" / "processed"
print("Input folder: insider-trading-phase3-4/data/processed")

Input folder: insider-trading-phase3-4/data/processed


### 2.2 Reading the Exported Tables

We read company CIKs as text to preserve their leading zeros and parse the event and filing dates as dates. The plan flag uses an integer type that allows missing values. The weekly return, membership, and cleaning-log files provide supporting information about how the event table was constructed.

In [2]:
# Read the event dataset and supporting exports with explicit identifier and date types.
df_events = pd.read_csv(processed_dir / "insider_events.csv",
                        dtype={"issuer_cik": "string", "any_plan_10b5_1": "Int64"},
                        parse_dates=["event_week", "first_filing"])
df_weekly = pd.read_csv(processed_dir / "weekly_stock_prices.csv", parse_dates=["week"])
df_cleaning_log = pd.read_csv(processed_dir / "cleaning_log.csv")
df_historical_clean = pd.read_csv(processed_dir / "historical_membership_clean.csv",
                                  parse_dates=["snapshot_date", "valid_from", "valid_to"])
print(f"Event table: {len(df_events):,} rows and {len(df_events.columns)} columns")
print(f"Weekly return table: {len(df_weekly):,} rows")
print(f"Historical membership: {len(df_historical_clean):,} intervals")
display(df_events.head())

Event table: 27,820 rows and 25 columns
Weekly return table: 219,378 rows
Historical membership: 628 intervals


,issuer_cik,issuer_name,sp_ticker,price_ticker,trans_code,event_week,first_filing,n_trades,n_insiders,total_value,log_value,role,role_rank,any_c_suite,any_plan_10b5_1,mixed_week,mean_filing_lag,alpha,beta,estimation_weeks,car_pre4,car_1w,car_4w,car_13w,car_26w
0,0000021665,COLGATE PALMOLIVE CO,CL,CL,S,2020-01-03,2020-01-02,1,1,1221022.50,14.015199,Other officer,2,0,<NA>,0,3.0,-0.000241,0.736484,51,-0.023257,0.024017,0.091272,0.189963,0.102511
1,0000051253,INTERNATIONAL FLAVORS & FRAGRANCES INC,IFF,IFF,P,2020-01-03,2020-01-02,2,1,8272210.00,15.928412,10% owner,4,0,<NA>,0,3.0,-0.005327,1.019928,51,-0.112750,-0.007962,0.062931,0.090631,0.209370
2,0000060086,LOEWS CORP,L,L,S,2020-01-03,2020-01-02,1,1,215379.50,12.280157,Other officer,2,0,<NA>,0,3.0,-0.001491,0.807514,51,0.033689,-0.019852,-0.017581,-0.230123,-0.315056
3,0000098246,TIFFANY & CO,TIF,NaN,S,2020-01-03,2020-01-02,7,7,91742269.60,18.334494,C-suite,1,1,<NA>,0,3.0,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
4,0000875320,VERTEX PHARMACEUTICALS INC / MA,VRTX,VRTX,S,2020-01-03,2020-01-02,4,1,6562715.24,15.696915,Director,3,0,<NA>,0,3.0,0.000093,1.034535,51,-0.053690,0.039644,0.044175,0.340251,0.302255


### 2.3 Checking the Imported Snapshot

The cleaning run produced 27,820 events and 25 columns. We check those dimensions, the event key, identifier formatting, and the fields needed for the two planned models. These checks stop the notebook if the exports do not match the documented snapshot. They do not delete, fill, or alter observations.

In [3]:
# Check the exported schema, event uniqueness, identifiers, and allowed transaction directions.
expected_columns = ["issuer_cik", "issuer_name", "sp_ticker", "price_ticker", "trans_code", "event_week", "first_filing",
                    "n_trades", "n_insiders", "total_value", "log_value", "role", "role_rank", "any_c_suite",
                    "any_plan_10b5_1", "mixed_week", "mean_filing_lag", "alpha", "beta", "estimation_weeks",
                    "car_pre4", "car_1w", "car_4w", "car_13w", "car_26w"]
assert df_events.columns.tolist() == expected_columns, "Review the export schema before continuing."
assert df_events.shape == (27820, 25), "The input snapshot changed; update its documentation before analysis."
assert df_events[["issuer_cik", "event_week", "trans_code"]].notna().all().all()
assert not df_events.duplicated(["issuer_cik", "event_week", "trans_code"]).any()
assert df_events["issuer_cik"].str.fullmatch(r"\d{10}").all()
assert set(df_events["trans_code"]) == {"P", "S"}
assert df_events["any_c_suite"].isin([0, 1]).all()
assert df_events["total_value"].gt(0).all()
assert np.allclose(df_events["log_value"], np.log(df_events["total_value"]))
print("Snapshot checks passed. No rows were changed or removed.")

Snapshot checks passed. No rows were changed or removed.


### 2.4 Coverage and Missing Values

With the files loaded, we check the sample sizes and missing values needed to describe the data. We count which events have the required fields for each original plan. These are eligibility checks, not comparisons of outcome values between groups. The full event table remains intact, and missing returns are not replaced with zero.

In [4]:
# Report coverage and planned-model eligibility without fitting models or comparing return values.
coverage = df_events.groupby("trans_code").agg(events=("issuer_cik", "size"), companies=("issuer_cik", "nunique"))
coverage.index = coverage.index.map({"P": "Purchases", "S": "Sales"})
display(coverage)
display(df_events.isna().sum().rename("missing_values").to_frame())
eligible_1 = df_events["trans_code"].isin(["P", "S"]) & df_events["car_13w"].notna()
eligible_2 = df_events["trans_code"].eq("P") & df_events[["car_13w", "any_c_suite", "log_value", "n_insiders"]].notna().all(axis=1)
print(f"Plan 1 eligible events: {eligible_1.sum():,}; excluded: {(~eligible_1).sum():,}")
print(f"Plan 2 eligible purchase events: {eligible_2.sum():,}")
print(f"Purchase events excluded from Plan 2: {(df_events['trans_code'].eq('P') & ~eligible_2).sum():,}")
print(f"Event weeks: {df_events['event_week'].min().date()} to {df_events['event_week'].max().date()}")
display(df_cleaning_log)

Plan 1 eligible events: 26,100; excluded: 1,720
Plan 2 eligible purchase events: 2,227
Purchase events excluded from Plan 2: 129
Event weeks: 2020-01-03 to 2026-01-02


,events,companies
trans_code,,
Purchases,2356,459
Sales,25464,582


,missing_values
issuer_cik,0
issuer_name,13
sp_ticker,0
price_ticker,1665
trans_code,0
event_week,0
first_filing,0
n_trades,0
n_insiders,0
total_value,0


,step,rows
0,Raw non-derivative transactions,1841674
1,1a. Forms 4 and 4/A only,1820329
2,1b. Purchase (P) and sale (S) codes only,711439
3,"1c. Valid dates (lag 0-365 days), shares > 0, ...",700020
4,1d. Superseded amendment rows removed,692025
5,2b. Company in S&P 500 on filing date,127529
6,2c. Same trade under two CIKs removed,127368
7,4a. Trades rolled up into company-week events ...,27820
8,4b. Events with a price series and >= 40 weeks...,26100


## 3. Data Description and Cleaning Summary

### Purpose

This dataset was assembled for the Cornell INFO 2950 group project by Vansh Kakra, Dev Shukla, and Wilson Wang. Its purpose is to examine whether public insider disclosures contain information associated with subsequent stock returns beyond market movements. This relates to **market efficiency**, the extent to which prices already reflect available information. It is intended for educational, observational analysis, not individual surveillance, investment recommendations, or detecting illegal trading. 

This description follows [Gebru et al. (2018), *Datasheets for Datasets*](https://arxiv.org/abs/1803.09010).

### Sample and Coverage

The analysis file, `data/processed/insider_events.csv`, contains **27,820 rows and 25 columns**. One observation is a unique **(issuer_cik, event_week, trans_code)** combination: one company, Friday-labeled week, and direction. A company with both buying and selling has two rows that week. The 593 companies are not 593 simultaneous index constituents; membership changes across the six-year filing period.

There are **2,356 purchase events in 459 companies** and **25,464 sale events in 582 companies**, derived from **6,358 purchase and 121,010 sale trade records**. Filing dates cover January 2, 2020 through December 31, 2025, while event weeks cover January 3, 2020 through January 2, 2026. The entry rule shifts some late-2025 filings into 2026. Selection is based on filings during the study period, not exclusively transaction dates during that period.

**Missing data:** 1,665 events lack a linked price ticker. Another 55 have a price ticker but no estimated beta, for **1,720 events without returns** in total. This leaves **26,100 events**, including 2,227 purchases and 23,873 sales, with all four CAR horizons. The 129 purchase events without returns comprise 114 without a price ticker and 15 additional events without beta. No extra 26-week attrition occurs in this saved snapshot. The plan flag is missing for **17,060 events**; missing is not coded as zero. Return exclusions are applied only when a particular analysis requires those values.

The interim files remain at a different unit: `sec_ps_trades.csv.gz` contains 692,025 cleaned SEC P/S trade rows across companies; `sp500_trades.csv.gz` contains the 127,368 matched S&P 500 trade rows. The membership and ticker-CIK files document matching. No train/test split is stored in the event file, and there are no hand-labeled outcomes: CARs are computed quantities.

### Event Variables

CSV has no enforced schema. Read CIKs as strings to retain leading zeros and parse date columns explicitly. Returns are stored as decimals; multiply by 100 to express percentage points. The following types describe intended values, including missing numeric values where stated.

| Column | Type | Meaning and units |
|---|---|---|
| `issuer_cik` | String | Ten-digit SEC company identifier, including leading zeros. |
| `issuer_name` | String | First retained company name within the event; names may change over time. |
| `sp_ticker` | String | First retained historical index ticker for the event. |
| `price_ticker` | String, nullable | Linked ticker used for prices, selected by latest membership end among available histories. |
| `trans_code` | Category | P for purchase or S for sale, as reported in Form 4. |
| `event_week` | Date | Friday on or after filing date plus one weekday; event aggregation and entry label. |
| `first_filing` | Date | Earliest filing date among trades in this event. |
| `n_trades` | Integer | Number of retained trade rows in the event, not number of filings. |
| `n_insiders` | Integer | Number of distinct retained owner CIKs; joint filings can limit interpretation. |
| `total_value` | Float | Sum of reported shares times price per share, in nominal dollars. |
| `log_value` | Float | Natural logarithm of total dollar value, dimensionless. |
| `role` | Category | Most senior assigned role under the project's ranking. |
| `role_rank` | Integer | 1 C-suite, 2 Other officer, 3 Director, 4 10% owner, 5 Other. |
| `any_c_suite` | Binary integer | 1 if any retained trade has assigned C-suite rank, otherwise 0. |
| `any_plan_10b5_1` | Nullable binary | Maximum observed filing plan flag within the event; missing if all flags are missing. A zero can coexist with missing component flags. |
| `mixed_week` | Binary integer | 1 if the same company has both P and S events that week. |
| `mean_filing_lag` | Float | Mean filing date minus transaction date over event trades, in calendar days. |
| `alpha` | Nullable float | Market-model intercept, in weekly decimal-return units. |
| `beta` | Nullable float | Market-model slope against SPY, unitless. |
| `estimation_weeks` | Integer | Number of available return rows considered, up to 52; 0 when no series exists, with at least 40 required for a fit. |
| `car_pre4` | Nullable float | Sum of four abnormal weekly returns ending with the event week; decimal units, not strictly pre-disclosure. |
| `car_1w` | Nullable float | Sum of abnormal returns over the first week after the event week, decimal units. |
| `car_4w` | Nullable float | Sum over the first 4 weeks after the event week, decimal units. |
| `car_13w` | Nullable float | Sum over the first 13 weeks after the event week, decimal units. |
| `car_26w` | Nullable float | Sum over the first 26 weeks after the event week, decimal units. |

### Source Coverage and People

The **SEC** is the primary source of legally mandated disclosures. Its records document reported transactions rather than verified motives or illegal conduct. **Historical membership** comes from a community-maintained resource that adds coverage of departed firms. **Yahoo Finance** supplies adjusted price histories, accessed through the third-party yfinance package. Each source supplies information the others lack: transactions, historical eligibility, and market outcomes, respectively.

Section 16 reporting obligations cover officers, directors, and beneficial owners above the relevant 10% threshold, not every employee or investor. Form 4 changes in ownership generally have a two-business-day reporting deadline. This creates a selected, self-reported sample with varying disclosure delays. See the [SEC investor bulletin](https://www.sec.gov/files/forms-3-4-5.pdf). The S&P 500 restriction further selects large companies. Vendor coverage, ticker changes, filing corrections, and title wording influence which observations survive matching.

The plan checkbox became required for applicable reports filed on or after **April 1, 2023**, according to the [SEC's 10b5-1 disclosure announcement](https://www.sec.gov/newsroom/modernizing-rule-10b5-1-insider-trading-plans). Earlier missingness cannot be interpreted as absence of a plan. Roles are derived from self-reported relationship fields and titles, not independently verified employment histories.

**People and awareness:** Filers provide identifying information for a public regulatory disclosure process. Public filing does not establish that they consented to this particular research use. We did not contact insiders or collect research consent, and cannot verify each person's understanding of downstream reuse. Raw files identify individuals and may include addresses; the final event table uses company identifiers and aggregated roles rather than insider names. Owner identifiers in interim data can still be linked to individuals. We avoid individual rankings or allegations.

### Preparation

The separate cleaning notebook uses pandas and NumPy to combine filings, membership, and prices. yfinance was used for the Phase 2 price download; we reused those saved prices.

The [cleaning notebook](https://github.com/Vansh-Kakra/DSproject/blob/phase-3-analysis/insider-trading-phase3-4/data_cleaning.ipynb) documents the selection rules, amendment handling, role assignment, historical membership matching, and price calculations. The final table combines trades by company, week, and direction and attaches the market-adjusted returns. Intermediate files preserve the trade-level records used to construct it.

### Access and Intended Use

Suitable uses include descriptive summaries and the limited course-method association models specified below. Unsuitable uses include inferring illegal trading, identifying individual wrongdoing, or presenting a guaranteed profitable strategy. The data do not contain experimental assignments or controls sufficient for causal claims. They are a fixed course-project snapshot, with no promised update schedule or independent audit of all source records.

Reviewers can obtain the [saved raw files](https://github.com/Vansh-Kakra/DSproject/tree/7f8759d9b4af42a03e0c08f3da9c2baccd938f72/insider-trading-phase2/data/raw), [intermediate files](https://github.com/Vansh-Kakra/DSproject/tree/7f8759d9b4af42a03e0c08f3da9c2baccd938f72/insider-trading-phase2/data/interim), and [processed files](https://github.com/Vansh-Kakra/DSproject/tree/7f8759d9b4af42a03e0c08f3da9c2baccd938f72/insider-trading-phase2/data/processed) from the submitted Phase 2 snapshot. The [Phase 3 exports](https://github.com/Vansh-Kakra/DSproject/tree/phase-3-analysis/insider-trading-phase3-4/data/processed) were regenerated by the separate cleaning notebook; the rebuilt event table matches the saved Phase 2 event values. Original sources are the [SEC quarterly datasets](https://www.sec.gov/data-research/sec-markets-data/insider-transactions-data-sets), [fja05680/sp500](https://github.com/fja05680/sp500), and [Yahoo Finance](https://finance.yahoo.com/) through [yfinance](https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html). We credit the regulator, community maintainers, price provider, and software contributors; inclusion in this educational repository does not imply ownership of their data or unrestricted redistribution rights.

## 4. Original Preregistrations

The next two statements reproduce the submitted Phase 2 plans verbatim, including their original section numbers. They record what was planned before the research models were fitted. The earlier EDA exposure remains disclosed in Statement 1.

The original statements exclude significance tests and specify a random event split. Those choices are preserved here as part of the record. Section 5 records the changes used for the time split and overlap checks and distinguishes the fitted models from the remaining significance evaluation.

## 7. Preregistration Statement 1

### Purchases and Subsequent Returns

**Prior analysis:** Both plans were written after the EDA showed separate purchase and sale return distributions, but before fitting the research models or comparing returns by role or purchase size. These are not outcome-blind plans. Any later changes will be identified in the final report.

**Hypothesis:** The average **13-week CAR after purchase events is positive and exceeds the average after sale events**. The primary horizon is 13 weeks; 1, 4, and 26 weeks provide secondary descriptive context.

**Motivation:** [Lakonishok and Lee (2001), *Are Insider Trades Informative?*](https://www.nber.org/papers/w6656) find purchases more informative than sales, with important firm-size differences. [Jeng, Metrick, and Zeckhauser (2003), *Estimating the Returns to Insider Trading: A Performance-Evaluation Perspective*](https://rzeckhauser.scholars.harvard.edu/publications/estimating-returns-insider-trading-performance-evaluation-perspective) report abnormal performance for purchase portfolios but not comparable evidence for sale portfolios. These findings motivate a directional hypothesis; their samples, weighting, and transaction timing differ from our public-disclosure event design and do not guarantee the result here.

### Sample

The unit is one **company-week-direction event**, equally weighted regardless of its dollar value. Eligible filings are retained Form 4/4-A P/S records from 2020-2025 at companies matched to S&P 500 membership on the filing date. P/S is the operational purchase/sale definition and can include private transactions. Entry uses the **first Friday close on or after filing date plus one business day**, as implemented by a weekday offset; holiday and filing-time limitations are documented above.

The market model estimates alpha and beta using up to **52 available weekly returns strictly before the event week**, requiring at least **40**. Weekly abnormal return is actual stock return minus the model prediction (alpha plus beta times SPY return), written as `actual - (alpha + beta * market_return)`. `car_13w` sums the next 13 weekly abnormal returns, excluding the event week. It is stored in decimal units.

Use events with observed `car_13w` and a P/S direction. The current eligible sample is **26,100 events: 2,227 purchases and 23,873 sales**. Exclude missing price histories, insufficient estimation history, and missing outcomes; do not impute zero returns. Keep `mixed_week = 1` events in the primary analysis. Do not trim or winsorize valid outcomes based on their value. Report the retained counts and exclusions.

### Planned Analysis

Fit ordinary least squares with an intercept using `sklearn.linear_model.LinearRegression`:

`car_13w = intercept + beta_purchase * is_purchase + error`

`is_purchase` is 1 for P and 0 for S, so **sales are the reference group**. The intercept estimates average sale CAR; `beta_purchase` estimates the purchase-minus-sale difference. The implied purchase mean is `intercept + beta_purchase`. The hypothesis predicts **both `beta_purchase > 0` and `intercept + beta_purchase > 0`**. Both conditions are needed to support the hypothesis.

Report both group means, the coefficient and implied purchase mean in percentage points, and purchase-only mean CARs at 1, 4, 13, and 26 weeks with their available counts. A decimal coefficient of 0.01 represents a one-percentage-point difference. **We will assess coefficient direction and size, without p-values or statistical-significance claims.** In particular, repeated events and overlapping windows do not justify treating every row as independent evidence.

**Sensitivity analysis:** Repeat the same 13-week model after excluding `mixed_week = 1` events, reporting the new counts and coefficient alongside the primary result. This sensitivity check does not fix other window overlap. Report positive, zero, negative, and inconsistent results, regardless of whether they support the hypothesis. A null or opposite sign will not trigger a switch of the primary horizon. Further specifications will be labeled exploratory.

## 8. Preregistration Statement 2

### Insider Role and Purchase Size

**Hypothesis:** Among purchase events, **C-suite participation and larger dollar purchase values are associated with higher 13-week CARs**, controlling for the number of recorded insiders.

**Motivation:** Executives may have broad access to company information, and committing more dollars may indicate stronger conviction, although wealth and liquidity can also explain purchase size. [Seyhun (1986), *Insiders' Profits, Costs of Trading, and Market Efficiency*](https://doi.org/10.1016/0304-405X(86)90060-7) motivates examining differences in information by insider position. [Cohen, Malloy, and Pomorski (2012), *Decoding Inside Information*](https://corpgov.law.harvard.edu/2012/02/03/decoding-inside-information/) distinguish routine trading from more informative trading. Their classification is not equivalent to our C-suite flag or 10b5-1 checkbox. The literature motivates heterogeneity, not certainty that our exact role proxy or larger nominal purchases will predict better outcomes. Jeng, Metrick, and Zeckhauser (2003) also find that top executives do not outperform other insiders in their setting, so the role hypothesis faces meaningful contrary evidence.

### Sample and Model

Use **purchase events only** (`trans_code = P`) with observed `car_13w`, `any_c_suite`, `log_value`, and `n_insiders`, currently **2,227 events**. Apply Statement 1's filing scope, event unit, entry rule, market model, and return definition. Exclude missing required values without imputation and report exclusions. Keep valid extreme values and mixed weeks in the primary model. Each event receives equal weight.

`any_c_suite` equals 1 when an event includes a trade assigned C-suite rank: an officer title matching CEO, CFO, COO, their expanded names, or President excluding Vice President. Zero includes directors, other officers, 10% owners, and Other; it is **not a directors-only reference group**. `log_value` is the natural logarithm of total reported purchase dollars. `n_insiders` counts distinct retained owner CIKs. Joint-filing and title limitations apply.

Fit the following multivariable linear regression with an intercept using `LinearRegression`:

`car_13w = intercept + beta_c_suite * any_c_suite + beta_log_value * log_value + beta_insiders * n_insiders + error`

The directional predictions are **`beta_c_suite > 0` and `beta_log_value > 0`**. Report the two predictions separately; support for one does not imply support for both. The C-suite coefficient is an adjusted difference in decimal CAR at the same recorded log value and insider count. For a 1% larger purchase, the exact modeled difference is `beta_log_value * log(1.01)`, approximately `0.01 * beta_log_value` in decimal CAR, holding other predictors fixed. Multiply by 100 to express percentage points. These are associations, not causal effects. No interactions, extra controls, or outcome-driven variable selection are included in the primary specification.

### Model Checks and Reporting

1. **Collinearity:** Report predictor Pearson/Spearman correlations and variance inflation factors (VIF). Compute each VIF as `1 / (1 - R_squared)` from regressing one predictor on the others with an intercept. Large values indicate overlapping predictor information. Discuss the diagnostics without automatically dropping a prespecified variable to improve its coefficient.
2. **Prediction evaluation:** Make one random **70% training / 30% test split**, using `train_test_split(test_size=0.30, random_state=2950)`. Fit the fixed specification on the training subset only. Evaluate once on the held-out test subset with **RMSE**, the square root of mean squared prediction error, and compare with a baseline that always predicts the training outcome mean. Report RMSE in percentage points and both sample sizes. Examine a training residual-versus-fitted plot for systematic patterns and unequal spread; do not tune the model on test outcomes.
3. **Association estimates:** Fit the same fixed specification on all eligible purchase events for descriptive coefficients, explicitly distinguishing that full-sample fit from the model used for held-out RMSE. Report every coefficient, its units, and the sign of each prespecified directional prediction. No t-tests, p-values, or statistical-significance claims are planned.
4. **Dependence caveat and sensitivity:** A random event split can place the same company and overlapping outcome periods in both subsets. Test RMSE therefore measures this random-split task and is not an estimate of independent future-period performance. Repeat the fixed full-sample association model after excluding mixed weeks, and present it beside the primary model. A time-ordered or company-grouped validation scheme is a reviewer question for Phase 3, not an undisclosed replacement chosen after results.

**Later-period extension:** Leave `any_plan_10b5_1` out of the main model. A separately labeled exploratory Phase 3 analysis may consider 2023-2025 filings with observed flags, starting no earlier than April 1, 2023 and checking component-flag coverage. The small positive-flag purchase sample limits precision. 

We will report results regardless of whether they support the hypotheses and document any changes to the specified model.

## 5. Phase 3 Analysis-Plan Update

**November 4, 2026. The rules below were recorded before fitting the research models.** The original Phase 2 statements above are preserved verbatim. Phase 2 EDA had already exposed some outcome distributions, so this is not an outcome-blind study.

### Decisions for This Analysis

We retain the two original linear regressions, the primary 13-week CAR, equal event weights, and the specified predictors. We do not trim valid extreme outcomes, impute missing returns, tune predictors on test results, or select a horizon based on its results. Purchase-only mean returns at 1, 4, and 26 weeks remain descriptive. The plan flag is not a predictor.

**Prediction split:** The purchase-only model trains on events before January 1, 2024 whose entire 13-week outcome window also ends before that date. The test set contains events on or after January 1, 2024 through the end of the saved filing-based sample, including event weeks in early January 2026 arising from late-2025 filings. Training candidates whose outcomes extend into the test period are excluded from training. We reconstruct actual outcome dates from the saved weekly return table rather than treating 13 rows as 13 calendar weeks. We fit the fixed model on training observations only and compare test RMSE with a constant prediction equal to the training mean. This replaces the original random 70/30 split, following Lecture 15's time-series validation and leakage precautions. No search over cutoffs or alternative models is planned. This assesses later-period performance for this company population, not generalization to entirely new companies.

**Sensitivity samples:** In addition to the full eligible sample, we retain the original check excluding mixed purchase/sale weeks. We add a stricter check: exclude mixed weeks, sort eligible events by company and event week, retain the earliest event, then retain the next event only if its entry week is on or after the last retained event's actual 13-week outcome end. Subsequent return windows therefore share no weeks within a company. We repeat this selection separately for the purchase/sale model and the purchase-only model. This avoids letting sale events displace purchase events in the purchase-only question. No outcome values determine which rows are selected. Counts and estimates for all three samples are reported. The stricter sample changes the composition and weighting of the data; it is not a replacement for the original full-sample estimate.

**Diagnostics:** We report purchase-model predictor correlations and VIF, a training residual-versus-fitted plot, and the fixed time-split prediction comparison. Full-sample association coefficients are distinct from the training-only prediction coefficients. None of these checks will trigger an unreported change in the model.

### Scope of the Results

These analyses estimate associations and prediction errors. The non-overlapping sample removes shared outcome weeks within a company, but repeated firms can still be related over time, and different companies can share shocks. Therefore, ordinary independent-row regression p-values, row bootstraps, or unrestricted permutations are not reported as valid significance tests. A time split addresses prediction leakage; it does not solve uncertainty for association estimates.

The significance section remains a separate task required by the rubric. We will make and document that methodological judgment ourselves; TA approval is not a prerequisite. An uncertainty method must state how it handles dependence and what assumptions remain. The family of substantive claims comprises the purchase mean, the purchase-minus-sale difference, and the C-suite and log-value coefficients. No significance claim will be made from coefficient signs alone. A multiple-testing correction cannot itself fix related observations.

### Course References

Lecture 14 covers prediction metrics; Lecture 15 covers time-aware evaluation and leakage; Lectures 16 and 17 cover regression inference, confidence intervals, and multiple testing. The time split and sensitivity selection here use pandas, NumPy, and the original scikit-learn regression methods. The Phase 3/4 rubric additionally requires a justified significance evaluation before the report is complete.

## 6. Analysis Samples and Outcome Windows

### 6.1 Recovering the Dates Used by Each Outcome

The cleaned event file contains CAR values but does not store the end date of each outcome window. We recover those dates from the weekly return export using the same next-13-observations rule as the cleaning notebook. We also verify that those observations reproduce every eligible stored 13-week CAR. This is an analysis-window check; the original cleaned CSV is not changed.

In [ ]:
# Recover the exact 13-week outcome windows and verify them against the exported CAR values.
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from IPython.display import Image
from io import BytesIO

weekly_by_ticker = {t: g.sort_values("week").reset_index(drop=True) for t, g in df_weekly.groupby("ticker")}
analysis_events = df_events.loc[eligible_1].copy()
window_start, window_end, rebuilt = [], [], []
for row in analysis_events.itertuples():
    series = weekly_by_ticker[row.price_ticker]
    position = series["week"].searchsorted(row.event_week, side="right")
    window = series.iloc[position:position + 13]
    assert len(window) == 13
    assert (window["week"].iloc[-1] - row.event_week).days <= 14 * 7
    window_start.append(window["week"].iloc[0])
    window_end.append(window["week"].iloc[-1])
    rebuilt.append((window["ret"] - (row.alpha + row.beta * window["mkt_ret"])).sum())
assert np.allclose(rebuilt, analysis_events["car_13w"], rtol=1e-9, atol=1e-10)
analysis_events["outcome_start"] = window_start
analysis_events["outcome_end"] = window_end
analysis_events["is_purchase"] = analysis_events["trans_code"].eq("P").astype(int)
predictors = ["any_c_suite", "log_value", "n_insiders"]
purchase_events = analysis_events.loc[analysis_events["trans_code"].eq("P") & analysis_events[predictors].notna().all(axis=1)].copy()
print(f"Verified actual outcome windows for {len(analysis_events):,} events; purchase-model sample: {len(purchase_events):,}.")

### 6.2 Selecting Events with Non-overlapping Outcomes

We first remove mixed purchase/sale weeks for this sensitivity check. Within each company, the earliest eligible event is retained. A later event is retained only when its entry week is on or after the previous retained event's outcome end. Since outcomes begin after entry, the selected outcome windows do not share return weeks. We apply the rule separately to the two model samples, check the resulting windows, and report how many events remain.

In [ ]:
# Build reproducible sensitivity samples without using return values to select events.
def select_nonoverlapping(frame):
    retained = []
    for _, group in frame.sort_values(["issuer_cik", "event_week"]).groupby("issuer_cik", sort=False):
        last_end = None
        for row in group.itertuples():
            if last_end is None or row.event_week >= last_end:
                retained.append(row.Index)
                last_end = row.outcome_end
    result = frame.loc[retained].sort_values(["issuer_cik", "event_week"]).copy()
    prior_end = result.groupby("issuer_cik")["outcome_end"].shift()
    assert (prior_end.isna() | result["outcome_start"].gt(prior_end)).all()
    assert not result.duplicated(["issuer_cik", "event_week"]).any()
    return result

samples_1 = {"All eligible": analysis_events,
             "No mixed weeks": analysis_events.loc[analysis_events["mixed_week"].eq(0)].copy()}
samples_1["No mixed weeks; non-overlapping"] = select_nonoverlapping(samples_1["No mixed weeks"])
samples_2 = {"All eligible": purchase_events,
             "No mixed weeks": purchase_events.loc[purchase_events["mixed_week"].eq(0)].copy()}
samples_2["No mixed weeks; non-overlapping"] = select_nonoverlapping(samples_2["No mixed weeks"])
sample_counts = pd.DataFrame([
    {"model": model, "sample": name, "events": len(frame), "companies": frame["issuer_cik"].nunique(),
     "purchases": int(frame["trans_code"].eq("P").sum()), "sales": int(frame["trans_code"].eq("S").sum())}
    for model, samples in [("Purchase versus sale", samples_1), ("Purchase role and size", samples_2)]
    for name, frame in samples.items()])
display(sample_counts)

## 7. Purchase Events and Subsequent Returns

### 7.1 The Primary 13-week Comparison

We fit the original regression with a purchase indicator and an intercept. The intercept is the sale mean, and the purchase coefficient is the purchase-minus-sale difference. Their sum is the purchase mean. We report these in percentage points for the full sample and both sensitivity samples. The first research hypothesis requires both a positive purchase mean and a positive purchase-minus-sale difference.

In [ ]:
# Fit the original purchase-versus-sale model and report descriptive estimates for all samples.
model1_rows = []
for name, frame in samples_1.items():
    fit = LinearRegression().fit(frame[["is_purchase"]], frame["car_13w"])
    sale_mean = frame.loc[frame["trans_code"].eq("S"), "car_13w"].mean()
    purchase_mean = frame.loc[frame["trans_code"].eq("P"), "car_13w"].mean()
    assert np.isclose(fit.intercept_, sale_mean)
    assert np.isclose(fit.intercept_ + fit.coef_[0], purchase_mean)
    model1_rows.append({"sample": name, "events": len(frame),
                        "purchase_mean_pp": purchase_mean * 100, "sale_mean_pp": sale_mean * 100,
                        "purchase_minus_sale_pp": fit.coef_[0] * 100})
model1_results = pd.DataFrame(model1_rows)
display(model1_results.round(4))

### 7.2 Other Return Horizons

The original plan also calls for purchase-only averages at 1, 4, 13, and 26 weeks. These are descriptive summaries using all purchase events with the corresponding outcome available. We keep 13 weeks as the primary horizon regardless of the other averages; these values are not additional significance tests.

In [ ]:
# Summarize the prespecified purchase-return horizons without choosing a new primary horizon.
purchase_horizons = pd.DataFrame([
    {"weeks": h, "available_purchase_events": int(df_events.loc[df_events["trans_code"].eq("P"), f"car_{h}w"].notna().sum()),
     "mean_car_pp": df_events.loc[df_events["trans_code"].eq("P"), f"car_{h}w"].mean() * 100}
    for h in [1, 4, 13, 26]])
display(purchase_horizons.round(4))

## 8. Insider Role and Purchase Size

### 8.1 Full-sample Association Estimates

Among purchases, we regress 13-week CAR on C-suite participation, the natural log of total purchase dollars, and the number of recorded insiders. Each coefficient describes an association holding the other two predictors fixed. Non-C-suite events are the reference for the C-suite indicator, not directors alone.

The table expresses coefficients in percentage points: per indicator change for C-suite participation, per one-unit change in natural log dollars, and per additional recorded insider. We also translate the log-value coefficient into the association with a 1% larger purchase. Full-sample estimates describe associations; the later prediction evaluation uses a separate training-only fit.

In [ ]:
# Fit the specified purchase-only regression without adding predictors or selecting observations by outcomes.
model2_rows = []
model2_fits = {}
for name, frame in samples_2.items():
    fit = LinearRegression().fit(frame[predictors], frame["car_13w"])
    model2_fits[name] = fit
    model2_rows.append({"sample": name, "events": len(frame), "intercept_pp": fit.intercept_ * 100,
                        "c_suite_pp": fit.coef_[0] * 100, "log_value_pp": fit.coef_[1] * 100,
                        "additional_insider_pp": fit.coef_[2] * 100,
                        "one_percent_larger_value_pp": fit.coef_[1] * np.log(1.01) * 100})
model2_results = pd.DataFrame(model2_rows)
display(model2_results.round(5))

### 8.2 Predictor Correlations and VIF

Correlations describe how the predictors move together. VIF measures how well each predictor can be explained by the other predictors: a large VIF indicates overlapping information. We calculate the original plan's Pearson and Spearman correlations and VIF on the full eligible purchase sample. These diagnostics do not trigger removal of a prespecified predictor.

In [ ]:
# Calculate the prespecified predictor correlations and variance inflation factors.
pearson = purchase_events[predictors].corr(method="pearson")
spearman = purchase_events[predictors].corr(method="spearman")
vif_rows = []
for target in predictors:
    others = [column for column in predictors if column != target]
    auxiliary = LinearRegression().fit(purchase_events[others], purchase_events[target])
    r_squared = auxiliary.score(purchase_events[others], purchase_events[target])
    vif_rows.append({"predictor": target, "VIF": 1 / (1 - r_squared)})
vif_results = pd.DataFrame(vif_rows)
print("Pearson correlations")
display(pearson.round(4))
print("Spearman correlations")
display(spearman.round(4))
display(vif_results.round(4))

## 9. Prediction on Later Events

### 9.1 A Calendar-based Split with an Outcome Gap

We use January 1, 2024 as the fixed boundary. Training events must occur before the boundary and have their full outcome observed before it. The test set begins at the boundary. We report removed boundary-crossing events and actual date ranges so that the split is auditable. Companies may appear in both sets, because this evaluation concerns later events in this company population rather than new-company prediction.

In [ ]:
# Separate earlier training events from later test events and exclude outcomes crossing the boundary.
cutoff = pd.Timestamp("2024-01-01")
train = purchase_events.loc[purchase_events["event_week"].lt(cutoff) & purchase_events["outcome_end"].lt(cutoff)].copy()
test = purchase_events.loc[purchase_events["event_week"].ge(cutoff)].copy()
boundary_excluded = purchase_events.loc[purchase_events["event_week"].lt(cutoff) & purchase_events["outcome_end"].ge(cutoff)].copy()
assert len(train) > 0 and len(test) > 0
assert train["outcome_end"].max() < test["event_week"].min()
assert set(train.index).isdisjoint(test.index)
assert len(train) + len(test) + len(boundary_excluded) == len(purchase_events)
split_table = pd.DataFrame([
    {"subset": name, "events": len(frame), "companies": frame["issuer_cik"].nunique(),
     "first_event": frame["event_week"].min(), "last_event": frame["event_week"].max(),
     "last_outcome": frame["outcome_end"].max()}
    for name, frame in [("Training", train), ("Boundary excluded", boundary_excluded), ("Test", test)]])
display(split_table)
print("Companies appearing in both training and test:", len(set(train["issuer_cik"]) & set(test["issuer_cik"])))

### 9.2 Comparing Predictions with a Simple Baseline

The fixed regression is fitted only on training events. The baseline predicts the training mean for every event. We compare RMSE, the typical prediction error measured as the square root of mean squared error, in percentage points. Lower RMSE is better. The training score describes fit to earlier data; the test score measures performance on later events. We will not change the predictors or cutoff in response to the test score.

In [ ]:
# Fit once on the training set and compare its errors with the training-mean baseline.
prediction_model = LinearRegression().fit(train[predictors], train["car_13w"])
training_mean = train["car_13w"].mean()
evaluation_rows = []
for name, frame in [("Training", train), ("Test", test)]:
    predicted = prediction_model.predict(frame[predictors])
    baseline = np.full(len(frame), training_mean)
    model_rmse = np.sqrt(mean_squared_error(frame["car_13w"], predicted)) * 100
    baseline_rmse = np.sqrt(mean_squared_error(frame["car_13w"], baseline)) * 100
    evaluation_rows.append({"subset": name, "events": len(frame), "model_RMSE_pp": model_rmse,
                            "baseline_RMSE_pp": baseline_rmse, "improvement_pp": baseline_rmse - model_rmse})
prediction_results = pd.DataFrame(evaluation_rows)
display(prediction_results.round(4))
training_coefficients = pd.DataFrame({"term": ["intercept"] + predictors,
                                      "coefficient_pp": np.r_[prediction_model.intercept_, prediction_model.coef_] * 100})
print("Training-only coefficients, separate from the full-sample association fit")
display(training_coefficients.round(4))

### 9.3 Training Residuals

A residual is the actual outcome minus the model's prediction. The plot below uses only training events. We look for systematic patterns, uneven spread, and extreme residuals. Valid extreme observations remain in the model, as specified in Phase 2. A quiet-looking plot would not establish independence or validate significance tests.

In [ ]:
# Plot training residuals against fitted outcomes without using test outcomes for model selection.
training_fitted = prediction_model.predict(train[predictors]) * 100
training_residual = train["car_13w"].to_numpy() * 100 - training_fitted
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(training_fitted, training_residual, s=16, alpha=0.4)
ax.axhline(0, color="black", linewidth=1)
ax.set(xlabel="Fitted 13-week CAR (percentage points)", ylabel="Residual (percentage points)",
       title="Purchase model: training residuals")
fig.tight_layout()
buffer = BytesIO()
fig.savefig(buffer, format="png", dpi=150)
display(Image(data=buffer.getvalue()))
plt.close(fig)
display(pd.Series(training_residual).describe(percentiles=[0.01, 0.5, 0.99]).rename("training_residual_pp").to_frame())

## 10. Interpretation and Remaining Work

The tables above report the fixed models, overlap sensitivities, and later-period prediction evaluation. Written interpretations will be completed after inspecting these results. Statistical significance remains unassessed; the non-overlapping sample and calendar split do not by themselves establish independence. The final report still needs a justified uncertainty analysis, completed conclusions, acknowledgements and bibliography, and the submission PDF.